# Entrainement et evaluation d'un model BERT "simpliste"

## Mise en place de l'environement de travail

In [1]:
import sys
import os

# Ajouter le chemin du répertoire racine du projet
project_root = os.path.abspath("..")
sys.path.append(project_root)
print(f"Racine du projet : {project_root}")

Racine du projet : /notebooks/Project


In [2]:
from config import DATA_DIR, UTILS_DIR

In [3]:
from utils.pipeline_bert import TextProcessingPipeline
import pandas as pd

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AdamW
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from utils.dataset import ToxicityDataset

## Pré-traitement des données

### Chargement du dataset d'entrainement en RAM

In [5]:
# Chargement du dtaframe d'entrainement en RAM
train_path = os.path.join(DATA_DIR, 'train.csv')
train = pd.read_csv(train_path)

In [6]:
print(train.head())

                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  


### Sous-étanchionnage des données non-toxiques

In [7]:
print(f"Répartition initiale des données du dataframe :\n{train['toxic'].value_counts()}")

Répartition initiale des données du dataframe :
toxic
0    144277
1     15294
Name: count, dtype: int64


In [8]:
# Nous équilibrons ici les données avant l'entrainement (autant de toxic que de non-toxiques)
# Taille cible pour les lignes où toxic == 0
sample_size = train['toxic'].sum()  # Nombre de lignes où toxic == 1

# Filtrer les lignes où toxic == 0
non_toxic = train[train['toxic'] == 0]

# Sous-échantillonnage aléatoire des lignes où toxic == 0
non_toxic = non_toxic.sample(n=sample_size, random_state=42)

# Conserver toutes les lignes où toxic == 1
toxic = train[train['toxic'] == 1]

# Combiner les sous-échantillons pour créer un nouveau dataframe
balanced_train = pd.concat([non_toxic, toxic], axis=0)

# Mélanger les lignes du dataframe final
balanced_train = balanced_train.sample(frac=1, random_state=42).reset_index(drop=True)

# Vérifier le résultat
print(balanced_train['toxic'].value_counts())

toxic
1    15294
0    15294
Name: count, dtype: int64


### Pré-traitement des données équilibrées

In [9]:
# OPTIONNEL - Nous pouvons ici choisir de ne conserver qu'une partie du dataframe pour aléger l'entrainement si l'on dispose de matérielle limité
balanced_train = balanced_train.head(1000)

In [10]:
# Initialisation de la pipeline
pipeline = TextProcessingPipeline()

# Exécution du pipeline complet
balanced_train = pipeline.run_pipeline(balanced_train)

# Affichage du résultat
print(balanced_train.head())

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

                 id                                       comment_text  toxic  \
0  1b7fe21f6b20f176  The Wack Pack \n\nYou're fucking insane and dr...      1   
1  38c211f6b4945ca9  Hornswaggle is a popular mis-spell, due to a J...      0   
2  626d12d910b34c40  You just revert my and a lot of peoples work, ...      1   
3  a3ab5b368eb6fde0  Happy anniversary, Frickative. Hope you arrive...      0   
4  06c9a0833ede4305  Note one previous comment: To my knowledge of ...      0   

   severe_toxic  obscene  threat  insult  identity_hate  \
0             0        1       0       1              0   
1             0        0       0       0              0   
2             0        1       0       1              0   
3             0        0       0       0              0   
4             0        0       0       0              0   

                                      processed_text  \
0  the wack pack youre fucking insane and drunk w...   
1  hornswaggle is a popular misspell, due to a

## Entrainement du model

### Mise en place de l'environement pytorch

In [11]:
model_name = "distilbert-base-uncased"

# Charger le modèle avec la configuration de quantification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 1,
    trust_remote_code=True,
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# Créer des étiquettes multi-étiquettes
label_columns = ['toxic']

# Split entre Train et Validation
train_df, test_df = train_test_split(balanced_train, test_size=0.2, random_state=42)

In [14]:
train_dataset = ToxicityDataset(train_df, label_columns)
test_dataset = ToxicityDataset(test_df, label_columns)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, pin_memory=True)

In [15]:
# Définir l'appareil (CPU ou GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Device : {device}")

Device : cuda


### Entrainement et evaluation du model

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = torch.nn.BCEWithLogitsLoss()  # Pour multi-label classification

In [17]:
def train(model, train_loader, optimizer, criterion, device, epochs=3):
    # Précision mixte avec AMP
    scaler = torch.cuda.amp.GradScaler()

    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            # Charger les données sur le GPU/CPU
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Remettre les gradients à zéro
            optimizer.zero_grad()

            # Utilisation de torch.cuda.amp pour le forward et le backward
            with torch.cuda.amp.autocast():
                # Forward pass
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=None)
                logits = outputs.logits

                # Calcul de la perte
                loss = criterion(logits, labels)

            # Backward pass avec AMP
            scaler.scale(loss).backward()

            # Optimisation avec mise à jour des gradients
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        print(f"Époque {epoch + 1}/{epochs}, Perte moyenne : {total_loss / len(train_loader):.4f}")

In [18]:
def evaluate(model, test_loader, device):
    model.eval()
    preds = []
    true_labels = []

    with torch.no_grad():
        for batch in test_loader:
            # Charger les données sur le GPU/CPU
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=None)
            logits = outputs.logits

            # Appliquer une sigmoïde pour les probabilités
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.extend(probs > 0.5)  # Seuil à 0.5
            true_labels.extend(labels.cpu().numpy())

    # Rapport de classification pour chaque étiquette
    print(f"Accuracy : {accuracy_score(true_labels, preds)}")
    print(f"Recall : {recall_score(true_labels, preds)}")
    print(f"Matrice de confusion :\n {confusion_matrix(true_labels, preds)}")
    

In [19]:
train(model, train_loader, optimizer, criterion, device, epochs=6)

Époque 1/6, Perte moyenne : 0.3656
Époque 2/6, Perte moyenne : 0.1634
Époque 3/6, Perte moyenne : 0.0701
Époque 4/6, Perte moyenne : 0.0476
Époque 5/6, Perte moyenne : 0.0238
Époque 6/6, Perte moyenne : 0.0109


In [21]:
# Évaluer le modèle
evaluate(model, test_loader, device)

Accuracy : 0.93
Recall : 0.896551724137931
Matrice de confusion :
 [[108   5]
 [  9  78]]


In [22]:
# Optionnel 
torch.save(model, "Bert_1000Samples_Blanced_Train_Basic.pt")